In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agents-mistral/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# Long-running agents — practice (Mistral edition)

Re-implement the methods that carry the five rules, then the two Mistral-specific pieces. Each exercise is graded by the real tests. Reference: `durable.py`, `mistral_model.py`, `mistral_workflow.py` (cover them up first).

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))                      # repo root (run from notebooks/)
from durable import Agent, Crash, FakeClock, FakeModel, LeaseHeld, PaymentAPI, Queue, Store, Wait
RUNS = "runs.json"

def fresh(script, tools=None, clock=None):
    if os.path.exists(RUNS): os.remove(RUNS)
    pay = PaymentAPI()
    return Agent(Store(RUNS), Queue(), FakeModel(script), tools or {"charge": pay}, clock=clock or FakeClock()), pay

def show(run):
    print(f"{run.id}  status={run.status}  waiting_on={run.waiting_on}  result={run.result}")
    for i, s in enumerate(run.journal):
        print(f"  [{i}] {s['type']:<8}", {k: v for k, v in s.items() if k != "type"})

import durable
sys.path.insert(0, os.path.abspath('../tests')); import test_core as T

## Exercise 1 — `Store.acquire_lease(run_id, owner, ttl, now)`  ·  rule (3)
Refuse if another owner holds an unexpired lease (raise `LeaseHeld`); otherwise set `run.lease = {"owner", "until"}` and save. Same owner may re-acquire.

In [ ]:
def acquire_lease(self, run_id, owner, ttl, now):
    run = self.get(run_id)
    # TODO
    raise NotImplementedError

durable.Store.acquire_lease = acquire_lease
T.test_two_workers_cannot_advance_the_same_run(); T.test_crash_after_side_effect_charges_once(); print("lease ✓")

## Exercise 2 — `Agent._decide(run)`  ·  rules (2) and (4)
1. If `run.decisions() >= run.max_steps` → `self._finish(run, "FAILED", "budget: ...")`, return `None`.
2. `d = self.model.decide(run.goal, run.journal)`; append `{"type": "decision", **d}`.
3. If `"final" in d` → `self._finish(run, "DONE", d["final"])`, return `None`.
4. Append the intent `{"type": "intent", "tool", "args", "key": f"{run.id}:{len(run.journal)}", "done": False, "approved": False}`, **save** (checkpoint 1), call `self._crash_maybe("after_intent")`, return the intent.

In [ ]:
def _decide(self, run):
    # TODO
    raise NotImplementedError

durable.Agent._decide = _decide
T.test_happy_path_one_step_per_wakeup(); T.test_budget_stops_a_runaway_loop(); T.test_crash_after_intent_before_side_effect_executes_once(); print("decide ✓")

## Exercise 3 — `Agent._execute(run, intent)`  ·  rules (2) and (5)
1. `tool = self.tools[intent["tool"]]`; if `tool.needs_approval` and not `intent["approved"]` → `return self._park(run, intent["key"], "approval")`.
2. `result = tool(**intent["args"], key=intent["key"])`, then `self._crash_maybe("after_side_effect")`.
3. If `result` is a `Wait` → `return self._park(run, result.token, "event")`.
4. Mark the intent done with the result, **save** (checkpoint 2), `self._wake(run)`, return the run.

In [ ]:
def _execute(self, run, intent):
    # TODO
    raise NotImplementedError

durable.Agent._execute = _execute
T.test_crash_after_side_effect_charges_once(); T.test_duplicate_delivery_is_ignored(); T.test_human_gate_parks_then_executes_exactly_what_was_approved(); T.test_slow_tool_parks_until_the_world_calls_back(); print("execute ✓")

## Exercise 4 — `Agent.resume(run_id, token, payload)`  ·  rule (5)
No-op unless the run is `WAITING` and the token matches. For `"approval"`: approved → `intent["approved"] = True`; rejected → mark the intent done with `{"rejected": reason}`. For `"event"`: mark the intent done with the payload. Then `RUNNING`, clear `waiting_on`, save, `_wake`.

In [ ]:
def resume(self, run_id, token, payload):
    # TODO
    raise NotImplementedError

durable.Agent.resume = resume
T.test_human_gate_parks_then_executes_exactly_what_was_approved(); T.test_rejection_is_recorded_and_the_loop_continues(); T.test_slow_tool_parks_until_the_world_calls_back(); print("resume ✓")

## Exercise 5 — `to_messages(goal, journal)`  ·  the Mistral adapter
Turn the journal into chat messages: `system` (instructions), `user` (goal), then for every tool decision an `assistant` message with one `tool_calls` entry `{"id", "type": "function", "function": {"name", "arguments": <json string>}}` followed by a `tool` message `{"tool_call_id", "name", "content": <json string of the intent's result>}`. In the journal a tool decision is always followed by its intent. Use `tool_call_id(intent["key"])` for the id (Mistral: 9 alphanumeric characters).

In [ ]:
import mistral_model
from mistral_model import tool_call_id, DEFAULT_INSTRUCTIONS
import test_mistral_model as TM

def to_messages(goal, journal, instructions=DEFAULT_INSTRUCTIONS):
    # TODO
    raise NotImplementedError

mistral_model.to_messages = to_messages
TM.test_journal_becomes_tool_calls_and_results(); TM.test_adapter_inside_the_durable_loop_survives_a_crash_without_reasking(); print("to_messages ✓")

## Exercise 6 — the workflow gate  ·  Mistral Workflows
Open `notebooks/practice_workflow.py` and fill in the gated branch of `MyInvoiceAgent.run`: reset `self.decision`, `await workflow.wait_condition(lambda: self.decision is not None, timeout=timedelta(seconds=approval_timeout_s))` inside a `try` that turns `asyncio.TimeoutError` into `{"status": "FAILED", "result": "approval expired", "journal": journal}`; on rejection record `{"rejected": reason}` on the intent and `continue`.

Workflow code lives in a module because the worker sandbox re-imports it by name. The grader runs your workflow on a local Temporal dev server (downloaded on first use; ~10 s).

Note the skeleton raises `workflows.WorkflowError`, not `NotImplementedError`: an *unexpected* exception in workflow code fails only the workflow **task**, and the platform retries the task until the code is fixed and redeployed — a bug never loses a run, but a grader would wait forever. Fail an execution deliberately with `WorkflowError`.

In [ ]:
import asyncio, importlib
import mistral_workflow as mw
import practice_workflow
from local_temporal import local_worker, TASK_QUEUE

async def grade():
    importlib.reload(practice_workflow)                    # pick up your edits
    mw.MODEL = FakeModel([{"tool": "charge", "args": {"amount": 42}}, {"final": "charged"}, {"tool": "charge", "args": {"amount": 42}}, {"final": "ok"}])
    mw.PAYMENTS = PaymentAPI()
    async with local_worker([practice_workflow.MyInvoiceAgent], [mw.decide, mw.charge]) as client:
        h = await client.start_workflow("invoice-agent-practice", {"goal": "pay 42"}, id="practice-approve", task_queue=TASK_QUEUE)
        await asyncio.sleep(0.5)
        try:
            await h.signal("decide", {"approved": True})
        except Exception:
            pass                                            # already finished → the result explains why
        r = (await result_or_hint(h))["result"]
        assert r["status"] == "DONE" and list(mw.PAYMENTS.charges.values()) == [42.0], r
        h = await client.start_workflow("invoice-agent-practice", {"goal": "pay 42", "approval_timeout_s": 1}, id="practice-expire", task_queue=TASK_QUEUE)
        r = (await result_or_hint(h))["result"]
        assert r["status"] == "FAILED" and r["result"] == "approval expired", r
    return "workflow gate ✓"

async def result_or_hint(handle):
    try:
        return await handle.result()
    except Exception as e:
        raise AssertionError(f"the workflow failed: {e.__cause__ or e}") from None

print(await grade())

### Reveal the reference gate

In [ ]:
import inspect
src = inspect.getsource(mw)
start, end = src.index("if d[" + chr(34) + "tool" + chr(34) + "] in gated_tools"), src.index("result = await TOOLS")
print(src[start:end])

## Think about it
1. Which crash window is still open after both checkpoints in `durable.py`? What closes it — the key, the memo, or both? Where does the same window sit in the Workflow version?
2. On Mistral Workflows the model call is an activity. If it fails with a 429 it is retried — fine. If the *worker* dies after the model answered but before the result was recorded, what happens, and why is that safe here but not in a naive loop?
3. A GSI partner wants the orchestrator on-prem. Which parts of `mistral_workflow.py` change, and which don't?

In [ ]:
answers = '''\n1.\n2.\n3.\n'''